<a href="https://colab.research.google.com/github/Requireding/AI-Research-Paper-Intelligence-System/blob/main/AI_Research_Paper_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers",split = 'train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers",split = 'train')

df = pd.DataFrame(dataset)
df = df[['title', 'abstract']].copy()
display(df.head())

In [ ]:
df = pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"] = df["title"] + "" + df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df["paper_text"])

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
df["paper_text"] = df["paper_text"].fillna("").astype(str)
df["paper_text"] = df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [ ]:
embedding =model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).tolist())

In [ ]:
print(sample_embedding.shape)

In [ ]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
for i in range(1, 5):
    sim = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[i].reshape(1, -1))
    print(f"Similarity with Paper {i}: {sim[0][0]}")

In [ ]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
  embeddings = np.load("paper_embeddings.npy")
else:
  print("Generating embeddings")
  embeddings = model.encode(df["paper_text"].tolist(),batch_size=32,show_progress_bar=True)
  np.save("paper_embeddings.npy",embeddings)
  print("Embeddings saved")

In [ ]:
print("Cleaned DataFrame saved successfully!")

In [ ]:
query = "deep learning for medical image analysis"
query_emb = model.encode([query])

In [ ]:
similarities = cosine_similarity(query_emb, sample_embedding)

print("Similarity scores for the first 5 papers:\n", similarities)

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

In [ ]:
papers_to_encode = df["paper_text"].tolist()[:50000]

In [ ]:
print("Encoding 50,000 papers (Watch the green bar below)...")
all_embeddings = model.encode(papers_to_encode, show_progress_bar=True)

In [ ]:
faiss_embeddings = np.array(all_embeddings).astype('float32')
faiss.normalize_L2(faiss_embeddings)

In [ ]:
index = faiss.IndexFlatIP(384)
index.add(faiss_embeddings)

In [ ]:
query_embedding = model.encode([query]).astype('float32')
faiss.normalize_L2(query_embedding)

k = 5 # Number of top results we want
distances, indices = index.search(query_embedding, k)

print("\n--- Search Results ---")
print("Top 5 distances:", distances)
print("Top 5 dataframe indices:", indices)

In [ ]:
print("\nTop Matching Papers:")
for i in indices[0]:
    print("-", df.iloc[i]['title'])

In [ ]:
import faiss

In [ ]:
faiss.normalize_L2(embedding.reshape(1, -1))

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding.reshape(1, -1))

In [ ]:
import faiss

# Re-create the index using the already available faiss_embeddings (which contain 50,000 items)
# This addresses the issue where the index was re-initialized in a subsequent cell,
# ensuring it holds 50,000 items before printing ntotal.
index = faiss.IndexFlatIP(faiss_embeddings.shape[1]) # Use the dimension from faiss_embeddings
index.add(faiss_embeddings)
print(index.ntotal)

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
query_embedding = model.encode([query])
query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

In [ ]:
faiss.normalize_L2(query_embedding)
query_embedding

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df.iloc[25846]["title"])

In [ ]:
print(df.iloc[25846]["abstract"])

In [ ]:
print(df.iloc[37567]["title"])

In [ ]:
print(df.iloc[37567]["abstract"])

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
D, I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
def search_paper(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)
  for score,idx  in zip(D[0],I[0]) :
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()
  return D, I

In [ ]:
search_paper("deep learning for medical image analysis")

In [ ]:
!pip install transformers==4.46.3

In [ ]:
import transformers
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
type(summarizer)

In [ ]:
import pandas as pd
from datasets import load_dataset

In [ ]:
# Ensure df is defined in case of a kernel restart or out-of-order execution
if 'df' not in locals():
    print("Initializing df...")
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']].copy()
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].fillna("").astype(str)
    df["paper_text"] = df["paper_text"].str.replace("\n"," ",regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("df initialized.")


In [ ]:
summary = summarizer(df.iloc[15666]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
type(summary)

In [ ]:
summary[0]["summary_text"]

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import faiss

In [ ]:
import numpy as np

In [ ]:
# Ensure 'paper_text' column exists and is cleaned
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].fillna("").astype(str)
df["paper_text"] = df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [ ]:
# Initialize the sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Prepare embeddings for FAISS index
papers_to_encode = df["paper_text"].tolist()[:15000] # Use only first 15000 papers as per previous context
all_embeddings = model.encode(papers_to_encode, show_progress_bar=False) # No progress bar in programmatic fix
faiss_embeddings = np.array(all_embeddings).astype('float32')
faiss.normalize_L2(faiss_embeddings)

In [ ]:
# Initialize and add embeddings to FAISS index
index = faiss.IndexFlatIP(faiss_embeddings.shape[1])
index.add(faiss_embeddings)

In [ ]:
def search_and_summarize(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)

  print("\n--- Search Results ---")
  print(f"Query: {query}")
  print("\nTop Matching Papers:")
  for score,idx  in zip(D[0],I[0]) :
    print(f"Similarity score: {score:.4f}") # Format score for readability
    print("Title:",df.iloc[idx]["title"])
    print("Abstract:",df.iloc[idx]["abstract"][:5000]) # Print first 5000 chars

    # Generate summary
    summary_output = summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print("Summary:", summary_output[0]["summary_text"])
    print("-" * 30) # Separator for readability
  return D, I # Return distances and indices if needed elsewhere

In [ ]:
# Call the function
search_and_summarize("deep learning for medical image analysis", k=5)

In [ ]:
pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT
kw_model =KeyBERT(model)

In [ ]:
type(kw_model) #model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
text=df.iloc[25846]["abstract"]
keywords = kw_model.extract_keywords(text)

In [ ]:
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

In [ ]:
keywords=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english')

In [ ]:
print(keywords)

In [ ]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english', highlight=True)
finalkeyword